In [1]:
!pip install datasets==3.6.0
!pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 2.3 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 3.9 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import torch
import transformers
import torch.nn as nn
import torch.optim as optim
from transformers import AutoModel, BertTokenizer, BertConfig, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from pathlib import Path
from datasets import load_dataset, Dataset
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from transformers import TrainerCallback

from transformers.activations import ACT2FN
from transformers.cache_utils import Cache
from transformers.modeling_layers import GradientCheckpointingLayer
from transformers.modeling_outputs import BaseModelOutputWithPastAndCrossAttentions
from transformers.processing_utils import Unpack
from transformers.pytorch_utils import apply_chunking_to_forward
from transformers.utils import TransformersKwargs, auto_docstring
from transformers.models.bert.modeling_bert import BertAttention, BertModel, BertSelfAttention, BertLayer, BertPreTrainedModel, BertForSequenceClassification

device = torch.accelerator.current_accelerator() if torch.accelerator.is_available else "cpu"

config = BertConfig(
    vocab_size=130000,
    num_hidden_layers=8,
    hidden_size=512,
    num_attention_heads=8,
    max_position_embeddings=1024,
    num_labels=2,
)

dataset = load_dataset("RussianNLP/russian_super_glue", "russe", trust_remote_code=True)
df = dataset['train'].to_pandas()
train_df = df.iloc[:-1000].reset_index(drop=True)
val_df   = df.iloc[-1000:].reset_index(drop=True)
train_dataset = Dataset.from_pandas(train_df)
val_dataset   = Dataset.from_pandas(val_df)

train_dataloader = DataLoader(train_dataset, 50, 1)
val_dataloader = DataLoader(val_dataset, 50, 1)
print(train_dataset)

loss_fn = nn.CrossEntropyLoss()

In [3]:
import random

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [5]:
import matplotlib.pyplot as plt
import seaborn as sns

def angular_distance(x_l, x_ln):
    if isinstance(x_l, torch.Tensor):
        x_l = x_l.detach().cpu().numpy()
    if isinstance(x_ln, torch.Tensor):
        x_ln = x_ln.detach().cpu().numpy()

    x_l_flat = x_l.reshape(-1)
    x_ln_flat = x_ln.reshape(-1)

    dot = np.dot(x_l_flat, x_ln_flat)
    norm_l = np.linalg.norm(x_l_flat)
    norm_ln = np.linalg.norm(x_ln_flat)

    if norm_l == 0 or norm_ln == 0:
        return 1.0

    cos_sim = np.clip(dot / (norm_l * norm_ln), -1.0, 1.0)
    angle = np.arccos(cos_sim) / np.pi
    return angle


def layer_distance_matrix(hidden_states):
    n_layers = len(hidden_states)
    dist_matrix = np.zeros((n_layers, n_layers))

    for ii in range(n_layers):
        for jj in range(ii + 1, n_layers):
            dist = angular_distance(hidden_states[ii], hidden_states[jj])
            dist_matrix[ii, jj] = dist
            dist_matrix[jj, ii] = dist
        dist_matrix[ii, ii] = 0.0

    return dist_matrix

def get_all_layer_hidden_states(model, sentences1, sentences2, tokenizer):
    model.eval()

    encodings = tokenizer(
        sentences1,
        sentences2,
        padding=True,
        truncation=True,
        max_length=500,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.bert(
            **encodings,
            output_hidden_states=True,
            return_dict=True
        )
        hidden_states = outputs.hidden_states

    return list(hidden_states)

def matr_processing(model, tokenizer, sentences1, sentences2, file_name):
    all_layer_states = get_all_layer_hidden_states(model, sentences1, sentences2, tokenizer)

    dist_matrix = layer_distance_matrix(all_layer_states)

    plt.figure(figsize=(10, 8))
    sns.heatmap(
        dist_matrix,
        annot=True,
        fmt=".3f",
        cmap="viridis",
        square=True,
        xticklabels=range(dist_matrix.shape[0]),
        yticklabels=range(dist_matrix.shape[0]),
        cbar_kws={"shrink": 0.8, "label": "Угловое расстояение"}
    )
    plt.xlabel("Индекс слоя")
    plt.ylabel("Индекс слоя")
    plt.tight_layout()
    plt.savefig(f'/content/drive/MyDrive/colab_results/{file_name}_matr', dpi=300, bbox_inches='tight')
    plt.close()

    diag = np.diag(dist_matrix, k=1)
    mean = np.mean(diag)
    with open(f'/content/drive/MyDrive/colab_results//{file_name}_mean.txt', 'w', encoding='utf-8') as f:
        f.write(str(mean))

In [6]:
from collections import defaultdict

def compute_layer_gradient_norms(model, loss):
    if not any(p.grad is not None for p in model.parameters()):
        loss.backward(retain_graph=True)

    layer_norms = {}
    layer_grads = defaultdict(list)

    for name, param in model.named_parameters():
        if param.grad is None:
            continue

        if 'bert.encoder.layer' in name:
            layer_num = int(name.split('layer.')[1].split('.')[0])
            layer_key = f'encoder.layer_{layer_num}'
        else:
            continue

        layer_grads[layer_key].append(param.grad.reshape(-1))

    for layer_key, grads in layer_grads.items():
        all_grad = torch.cat(grads)
        norm = torch.norm(all_grad, p=2).item()
        layer_norms[layer_key] = norm

    return layer_norms

def plot_gradient_norms(layer_norms, file_name):
    layers = list(layer_norms.keys())
    norms = list(layer_norms.values())

    plt.figure(figsize=(12, 6))
    bars = plt.bar(layers, norms, color='skyblue', edgecolor='navy', alpha=0.8)

    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height * 1.01,
                 f'{height:.2f}', ha='center', va='bottom', fontsize=9)

    plt.xlabel("Слой", fontsize=12)
    plt.ylabel("Норма градиента", fontsize=12)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig(f'/content/drive/MyDrive/colab_results/{file_name}_hist', dpi=300, bbox_inches='tight')
    plt.close()

def grad_processing(model, tokenizer, sentences1, sentences2, labels, file_name):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

    encodings = tokenizer(sentences1, sentences2, padding=True, truncation=True,
                        max_length=500, return_tensors="pt")
    labels = torch.tensor(labels, dtype=torch.long)

    encodings = {k: v.to(device) for k, v in encodings.items()}
    labels = labels.to(device)

    outputs = model(**encodings, labels=labels)
    loss = outputs.loss

    layer_norms = compute_layer_gradient_norms(model, loss)

    norms = list(layer_norms.values())
    grad_var = np.var(norms)
    with open(f'/content/drive/MyDrive/colab_results//{file_name}_var.txt', 'w', encoding='utf-8') as f:
        f.write(str(grad_var))
    print(f"Дисперсия: {grad_var:.4f}")

    plot_gradient_norms(layer_norms, file_name)

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
test_sentence1 = []
for ii in range(15):
    test_sentence1.append(f"{train_dataset[ii]['word']} [SEP] {train_dataset[ii]['sentence1']}")
test_sentence2 = train_dataset[0:15]['sentence2']
test_label = train_dataset[0:15]['label']
print(test_sentence1)
print(test_sentence2)
print(test_label)

['двор [SEP] В нашей деревне осталось от силы двадцать дворов', 'доклад [SEP] Табличка на дверях: «Без доклада не входить»', 'засада [SEP] У нас вообще […] засада с героями, способными дотягивать в жизни до собственного творчества', 'доля [SEP] Он не успел сказать и десятой доли того, что собирался', 'закат [SEP] Теперь, если она не пойдет звонить мужу, успеет до заката к морю', 'демократия [SEP] Европейские демократии', 'заря [SEP] Заря новой эры', 'задержка [SEP] Задержка взгляда на собеседнике', 'дорожка [SEP] Бурые ковровые дорожки заглушали шаги', 'задержка [SEP] Вскоре я почувствовал всю муторность этой процедуры [сбора финансовых документов о тратах] и проклял все на свете, поскольку за несколько дней вынужденной задержки в Катманду, кажется, только этим и занимался', 'дух [SEP] Завертелась в доме веселая коловерть: праздничный стол, праздничный дух, шумные разговоры', 'защита [SEP] Однажды засыпая, погружаясь в замедленный, чуткий сон, Иван Макаров неожиданно понял, что единств

In [10]:
model = BertForSequenceClassification(config).to(device)
tokenizer = AutoTokenizer.from_pretrained("DeepPavlov/rubert-base-cased")
optimizer = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-2)

train_model(model, tokenizer, "russe/bert", train_dataloader, optimizer, loss_fn)

KeyboardInterrupt: 

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("DeepPavlov/rubert-base-cased")
model = AutoModelForSequenceClassification.from_pretrained("DeepPavlov/rubert-base-cased", num_labels=2).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-2)

train_model(model, tokenizer, "russe/rubert", train_dataloader, optimizer, loss_fn)

In [11]:
from transformers.activations import ACT2FN
from transformers.cache_utils import Cache
from transformers.modeling_layers import GradientCheckpointingLayer
from transformers.modeling_outputs import BaseModelOutputWithPastAndCrossAttentions
from transformers.processing_utils import Unpack
from transformers.pytorch_utils import apply_chunking_to_forward
from transformers.utils import TransformersKwargs, auto_docstring
from transformers.modeling_utils import ALL_ATTENTION_FUNCTIONS
from transformers.models.bert.modeling_bert import eager_attention_forward, BertAttention, BertModel, BertSelfAttention, BertEncoder, BertSelfOutput, BertIntermediate, BertOutput, BertLayer, BertPreTrainedModel, BertForSequenceClassification

class BertSelfAttentionPreLayerNorm(nn.Module):
    def __init__(self, config, is_causal=False, layer_idx=None):
        super().__init__()
        if config.hidden_size % config.num_attention_heads != 0 and not hasattr(config, "embedding_size"):
            raise ValueError(
                f"The hidden size ({config.hidden_size}) is not a multiple of the number of attention "
                f"heads ({config.num_attention_heads})"
            )
        self.config = config

        self.num_attention_heads = config.num_attention_heads
        self.attention_head_size = int(config.hidden_size / config.num_attention_heads)
        self.all_head_size = self.num_attention_heads * self.attention_head_size
        self.scaling = self.attention_head_size**-0.5

        self.query = nn.Linear(config.hidden_size, self.all_head_size)
        self.key = nn.Linear(config.hidden_size, self.all_head_size)
        self.value = nn.Linear(config.hidden_size, self.all_head_size)

        self.dropout = nn.Dropout(config.attention_probs_dropout_prob)

        self.is_decoder = config.is_decoder
        self.is_causal = is_causal
        self.layer_idx = layer_idx
        self.LayerNorm = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: torch.FloatTensor | None = None,
        past_key_values: Cache | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> tuple[torch.Tensor]:
        input_shape = hidden_states.shape[:-1]
        hidden_shape = (*input_shape, -1, self.attention_head_size)
        hidden_states = self.LayerNorm(hidden_states)

        query_layer = self.query(hidden_states).view(*hidden_shape).transpose(1, 2)
        key_layer = self.key(hidden_states).view(*hidden_shape).transpose(1, 2)
        value_layer = self.value(hidden_states).view(*hidden_shape).transpose(1, 2)

        if past_key_values is not None:
            current_past_key_values = past_key_values
            if isinstance(past_key_values, EncoderDecoderCache):
                current_past_key_values = past_key_values.self_attention_cache

            key_layer, value_layer = current_past_key_values.update(key_layer, value_layer, self.layer_idx)

        attention_interface: Callable = ALL_ATTENTION_FUNCTIONS.get_interface(
            self.config._attn_implementation, eager_attention_forward
        )

        attn_output, attn_weights = attention_interface(
            self,
            query_layer,
            key_layer,
            value_layer,
            attention_mask,
            dropout=0.0 if not self.training else self.dropout.p,
            scaling=self.scaling,
            **kwargs,
        )
        attn_output = attn_output.reshape(*input_shape, -1).contiguous()
        return attn_output, attn_weights

class BertSelfOutputPreLayerNorm(BertSelfOutput):
    def __init__(self, config):
        super().__init__(config)

    def forward(self, hidden_states: torch.Tensor, input_tensor: torch.Tensor) -> torch.Tensor:
        hidden_states = self.dense(hidden_states)
        hidden_states = self.dropout(hidden_states)
        hidden_states = hidden_states + input_tensor
        return hidden_states

class BertAttentionPreLayerNorm(BertAttention):
    def __init__(self, config, is_causal=False, layer_idx=None, is_cross_attention=False):
        super().__init__(config, is_causal=is_causal, layer_idx=layer_idx, is_cross_attention=False)
        self.self = BertSelfAttentionPreLayerNorm(config, is_causal=is_causal, layer_idx=layer_idx)
        self.output = BertSelfOutputPreLayerNorm(config)

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: torch.FloatTensor | None = None,
        encoder_hidden_states: torch.FloatTensor | None = None,
        encoder_attention_mask: torch.FloatTensor | None = None,
        past_key_values: Cache | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> tuple[torch.Tensor]:
        attention_mask = attention_mask if not self.is_cross_attention else encoder_attention_mask
        attention_output, attn_weights = self.self(
            hidden_states,
            encoder_hidden_states=encoder_hidden_states,
            attention_mask=attention_mask,
            past_key_values=past_key_values,
            **kwargs,
        )
        attention_output = self.output(attention_output, hidden_states)
        return attention_output, attn_weights

class BertIntermediatePreLayerNorm(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.LayerNorm = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)
        self.dense = nn.Linear(config.hidden_size, config.intermediate_size)
        if isinstance(config.hidden_act, str):
            self.intermediate_act_fn = ACT2FN[config.hidden_act]
        else:
            self.intermediate_act_fn = config.hidden_act

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        hidden_states = self.LayerNorm(hidden_states)
        hidden_states = self.dense(hidden_states)
        hidden_states = self.intermediate_act_fn(hidden_states)
        return hidden_states


class BertOutputPreLayerNorm(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.dense = nn.Linear(config.intermediate_size, config.hidden_size)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)

    def forward(self, hidden_states: torch.Tensor, input_tensor: torch.Tensor) -> torch.Tensor:
        hidden_states = self.dense(hidden_states)
        hidden_states = self.dropout(hidden_states)
        hidden_states = hidden_states + input_tensor
        return hidden_states


class BertLayerPreLayerNorm(BertLayer):
    def __init__(self, config, layer_idx=None):
        super().__init__(config)
        self.chunk_size_feed_forward = config.chunk_size_feed_forward
        self.seq_len_dim = 1
        self.attention = BertAttentionPreLayerNorm(config, is_causal=config.is_decoder, layer_idx=layer_idx)
        self.intermediate = BertIntermediatePreLayerNorm(config)
        self.output = BertOutputPreLayerNorm(config)

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: torch.FloatTensor | None = None,
        encoder_hidden_states: torch.FloatTensor | None = None,
        encoder_attention_mask: torch.FloatTensor | None = None,
        past_key_values: Cache | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> torch.Tensor:
        self_attention_output, _ = self.attention(
            hidden_states,
            attention_mask,
            encoder_hidden_states=encoder_hidden_states,
            encoder_attention_mask=encoder_attention_mask,
            past_key_values=past_key_values,
            **kwargs,
        )

        layer_output = apply_chunking_to_forward(
            self.feed_forward_chunk, self.chunk_size_feed_forward, self.seq_len_dim, self_attention_output
        )
        return layer_output

    def feed_forward_chunk(self, attention_output):
        intermediate_output = self.intermediate(attention_output)
        layer_output = self.output(intermediate_output, attention_output)
        return layer_output

class BertEncoderPreLayerNorm(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.layer = nn.ModuleList([BertLayerPreLayerNorm(config, layer_idx=i) for i in range(config.num_hidden_layers)])

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: torch.FloatTensor | None = None,
        encoder_hidden_states: torch.FloatTensor | None = None,
        encoder_attention_mask: torch.FloatTensor | None = None,
        past_key_values: Cache | None = None,
        use_cache: bool | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> tuple[torch.Tensor] | BaseModelOutputWithPastAndCrossAttentions:
        for layer_module in self.layer:
            hidden_states = layer_module(
                hidden_states,
                attention_mask,
                encoder_hidden_states=encoder_hidden_states,
                encoder_attention_mask=encoder_attention_mask,
                past_key_values=past_key_values,
                **kwargs,
            )

        return BaseModelOutputWithPastAndCrossAttentions(
            last_hidden_state=hidden_states,
            past_key_values=past_key_values if use_cache else None,
        )

class BertModelPreLayerNorm(BertModel):
    _no_split_modules = ["BertEmbeddings", "BertLayerPreLayerNorm"]
    _can_record_outputs = {
        "hidden_states": BertLayerPreLayerNorm,
        "attentions": BertSelfAttentionPreLayerNorm,
    }

    def __init__(self, config, add_pooling_layer=True):
        super().__init__(config, add_pooling_layer=add_pooling_layer)
        self.encoder = BertEncoderPreLayerNorm(config)

class BertForSequenceClassificationPreLayerNorm(BertForSequenceClassification):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.config = config

        self.bert = BertModelPreLayerNorm(config)
        classifier_dropout = (
            config.classifier_dropout if config.classifier_dropout is not None else config.hidden_dropout_prob
        )
        self.dropout = nn.Dropout(classifier_dropout)
        self.classifier = nn.Linear(config.hidden_size, config.num_labels)

        self.post_init()

In [12]:
class BertSelfAttentionHybridNorm(nn.Module):
    def __init__(self, config, is_causal=False, layer_idx=None):
        super().__init__()
        if config.hidden_size % config.num_attention_heads != 0 and not hasattr(config, "embedding_size"):
            raise ValueError(
                f"The hidden size ({config.hidden_size}) is not a multiple of the number of attention "
                f"heads ({config.num_attention_heads})"
            )
        self.config = config

        self.num_attention_heads = config.num_attention_heads
        self.attention_head_size = int(config.hidden_size / config.num_attention_heads)
        self.all_head_size = self.num_attention_heads * self.attention_head_size
        self.scaling = self.attention_head_size**-0.5

        self.query = nn.Linear(config.hidden_size, self.all_head_size)
        self.key = nn.Linear(config.hidden_size, self.all_head_size)
        self.value = nn.Linear(config.hidden_size, self.all_head_size)

        self.dropout = nn.Dropout(config.attention_probs_dropout_prob)

        self.is_decoder = config.is_decoder
        self.is_causal = is_causal
        self.layer_idx = layer_idx
        self.LayerNorm = nn.LayerNorm(self.attention_head_size, eps=config.layer_norm_eps)

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: torch.FloatTensor | None = None,
        past_key_values: Cache | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> tuple[torch.Tensor]:
        input_shape = hidden_states.shape[:-1]
        hidden_shape = (*input_shape, -1, self.attention_head_size)

        query_layer = self.query(hidden_states).view(*hidden_shape).transpose(1, 2)
        query_layer = self.LayerNorm(query_layer)
        key_layer = self.key(hidden_states).view(*hidden_shape).transpose(1, 2)
        key_layer = self.LayerNorm(key_layer)
        value_layer = self.value(hidden_states).view(*hidden_shape).transpose(1, 2)
        value_layer = self.LayerNorm(value_layer)

        if past_key_values is not None:
            current_past_key_values = past_key_values
            if isinstance(past_key_values, EncoderDecoderCache):
                current_past_key_values = past_key_values.self_attention_cache

            key_layer, value_layer = current_past_key_values.update(key_layer, value_layer, self.layer_idx)

        attention_interface: Callable = ALL_ATTENTION_FUNCTIONS.get_interface(
            self.config._attn_implementation, eager_attention_forward
        )

        attn_output, attn_weights = attention_interface(
            self,
            query_layer,
            key_layer,
            value_layer,
            attention_mask,
            dropout=0.0 if not self.training else self.dropout.p,
            scaling=self.scaling,
            **kwargs,
        )
        attn_output = attn_output.reshape(*input_shape, -1).contiguous()
        return attn_output, attn_weights

class BertAttentionHybridNorm(BertAttention):
    def __init__(self, config, is_causal=False, layer_idx=None, is_cross_attention=False):
        super().__init__(config, is_causal=is_causal, layer_idx=layer_idx, is_cross_attention=False)
        self.self = BertSelfAttentionHybridNorm(config, is_causal=is_causal, layer_idx=layer_idx)

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: torch.FloatTensor | None = None,
        encoder_hidden_states: torch.FloatTensor | None = None,
        encoder_attention_mask: torch.FloatTensor | None = None,
        past_key_values: Cache | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> tuple[torch.Tensor]:
        attention_mask = attention_mask if not self.is_cross_attention else encoder_attention_mask
        attention_output, attn_weights = self.self(
            hidden_states,
            attention_mask=attention_mask,
            past_key_values=past_key_values,
            **kwargs,
        )
        attention_output = self.output(attention_output, hidden_states)
        return attention_output, attn_weights

class BertLayerHybridNorm(BertLayer):
    def __init__(self, config, layer_idx=None):
        super().__init__(config)
        self.chunk_size_feed_forward = config.chunk_size_feed_forward
        self.seq_len_dim = 1
        self.attention = BertAttentionHybridNorm(config, is_causal=config.is_decoder, layer_idx=layer_idx)

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: torch.FloatTensor | None = None,
        encoder_hidden_states: torch.FloatTensor | None = None,
        encoder_attention_mask: torch.FloatTensor | None = None,
        past_key_values: Cache | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> torch.Tensor:
        self_attention_output, _ = self.attention(
        hidden_states,
        attention_mask,
        past_key_values=past_key_values,
        **kwargs,
    )
        attention_output = self_attention_output

        layer_output = apply_chunking_to_forward(
            self.feed_forward_chunk, self.chunk_size_feed_forward, self.seq_len_dim, attention_output
        )
        return layer_output

    def feed_forward_chunk(self, attention_output):
        intermediate_output = self.intermediate(attention_output)
        layer_output = self.output(intermediate_output, attention_output)
        return layer_output


class BertEncoderHybridNorm(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.layer = nn.ModuleList([BertLayerHybridNorm(config, layer_idx=i) for i in range(config.num_hidden_layers)])

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: torch.FloatTensor | None = None,
        encoder_hidden_states: torch.FloatTensor | None = None,
        encoder_attention_mask: torch.FloatTensor | None = None,
        past_key_values: Cache | None = None,
        use_cache: bool | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> tuple[torch.Tensor] | BaseModelOutputWithPastAndCrossAttentions:
        for layer_module in self.layer:
            hidden_states = layer_module(
                hidden_states,
                attention_mask,
                encoder_hidden_states=encoder_hidden_states,
                encoder_attention_mask=encoder_attention_mask,
                past_key_values=past_key_values,
                **kwargs,
            )

        return BaseModelOutputWithPastAndCrossAttentions(
            last_hidden_state=hidden_states,
            past_key_values=past_key_values if use_cache else None,
        )

class BertModelHybridNorm(BertModel):
    _no_split_modules = ["BertEmbeddings", "BertLayerHybridNorm"]
    _can_record_outputs = {
        "hidden_states": BertLayerHybridNorm,
        "attentions": BertSelfAttentionHybridNorm,
    }

    def __init__(self, config, add_pooling_layer=True):
        super().__init__(config, add_pooling_layer=add_pooling_layer)
        self.encoder = BertEncoderHybridNorm(config)

class BertForSequenceClassificationHybridNorm(BertForSequenceClassification):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.config = config

        self.bert = BertModelHybridNorm(config)
        classifier_dropout = (
            config.classifier_dropout if config.classifier_dropout is not None else config.hidden_dropout_prob
        )
        self.dropout = nn.Dropout(classifier_dropout)
        self.classifier = nn.Linear(config.hidden_size, config.num_labels)

        self.post_init()

In [13]:
class BertSelfAttentionScalingNorm(nn.Module):
    def __init__(self, config, is_causal=False, layer_idx=None):
        super().__init__()
        if config.hidden_size % config.num_attention_heads != 0 and not hasattr(config, "embedding_size"):
            raise ValueError(
                f"The hidden size ({config.hidden_size}) is not a multiple of the number of attention "
                f"heads ({config.num_attention_heads})"
            )
        self.config = config

        self.num_attention_heads = config.num_attention_heads
        self.attention_head_size = int(config.hidden_size / config.num_attention_heads)
        self.all_head_size = self.num_attention_heads * self.attention_head_size
        self.scaling = self.attention_head_size**-0.5

        self.query = nn.Linear(config.hidden_size, self.all_head_size)
        self.key = nn.Linear(config.hidden_size, self.all_head_size)
        self.value = nn.Linear(config.hidden_size, self.all_head_size)

        self.dropout = nn.Dropout(config.attention_probs_dropout_prob)

        self.is_decoder = config.is_decoder
        self.is_causal = is_causal
        self.layer_idx = layer_idx
        self.LayerNorm = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)

        self.attenuation_coef = 1/((layer_idx + 1) ** 0.5)

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: torch.FloatTensor | None = None,
        past_key_values: Cache | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> tuple[torch.Tensor]:
        input_shape = hidden_states.shape[:-1]
        hidden_shape = (*input_shape, -1, self.attention_head_size)
        hidden_states = self.LayerNorm(hidden_states) * self.attenuation_coef

        query_layer = self.query(hidden_states).view(*hidden_shape).transpose(1, 2)
        key_layer = self.key(hidden_states).view(*hidden_shape).transpose(1, 2)
        value_layer = self.value(hidden_states).view(*hidden_shape).transpose(1, 2)

        if past_key_values is not None:
            current_past_key_values = past_key_values
            if isinstance(past_key_values, EncoderDecoderCache):
                current_past_key_values = past_key_values.self_attention_cache

            key_layer, value_layer = current_past_key_values.update(key_layer, value_layer, self.layer_idx)

        attention_interface: Callable = ALL_ATTENTION_FUNCTIONS.get_interface(
            self.config._attn_implementation, eager_attention_forward
        )

        attn_output, attn_weights = attention_interface(
            self,
            query_layer,
            key_layer,
            value_layer,
            attention_mask,
            dropout=0.0 if not self.training else self.dropout.p,
            scaling=self.scaling,
            **kwargs,
        )
        attn_output = attn_output.reshape(*input_shape, -1).contiguous()
        return attn_output, attn_weights

class BertSelfOutputScalingNorm(BertSelfOutput):
    def __init__(self, config):
        super().__init__(config)

    def forward(self, hidden_states: torch.Tensor, input_tensor: torch.Tensor) -> torch.Tensor:
        hidden_states = self.dense(hidden_states)
        hidden_states = self.dropout(hidden_states)
        hidden_states = hidden_states + input_tensor
        return hidden_states

class BertAttentionScalingNorm(BertAttention):
    def __init__(self, config, is_causal=False, layer_idx=None, is_cross_attention=False):
        super().__init__(config, is_causal=is_causal, layer_idx=layer_idx, is_cross_attention=False)
        self.self = BertSelfAttentionScalingNorm(config, is_causal=is_causal, layer_idx=layer_idx)
        self.output = BertSelfOutputScalingNorm(config)

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: torch.FloatTensor | None = None,
        encoder_hidden_states: torch.FloatTensor | None = None,
        encoder_attention_mask: torch.FloatTensor | None = None,
        past_key_values: Cache | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> tuple[torch.Tensor]:
        attention_mask = attention_mask if not self.is_cross_attention else encoder_attention_mask
        attention_output, attn_weights = self.self(
            hidden_states,
            encoder_hidden_states=encoder_hidden_states,
            attention_mask=attention_mask,
            past_key_values=past_key_values,
            **kwargs,
        )
        attention_output = self.output(attention_output, hidden_states)
        return attention_output, attn_weights

class BertIntermediateScalingNorm(nn.Module):
    def __init__(self, config, layer_idx):
        super().__init__()
        self.LayerNorm = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)
        self.dense = nn.Linear(config.hidden_size, config.intermediate_size)

        self.attenuation_coef = 1 / ((layer_idx + 1) ** 0.5)

        if isinstance(config.hidden_act, str):
            self.intermediate_act_fn = ACT2FN[config.hidden_act]
        else:
            self.intermediate_act_fn = config.hidden_act

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        hidden_states = self.LayerNorm(hidden_states) * self.attenuation_coef
        hidden_states = self.dense(hidden_states)
        hidden_states = self.intermediate_act_fn(hidden_states)
        return hidden_states


class BertOutputScalingNorm(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.dense = nn.Linear(config.intermediate_size, config.hidden_size)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)

    def forward(self, hidden_states: torch.Tensor, input_tensor: torch.Tensor) -> torch.Tensor:
        hidden_states = self.dense(hidden_states)
        hidden_states = self.dropout(hidden_states)
        hidden_states = hidden_states + input_tensor
        return hidden_states


class BertLayerScalingNorm(BertLayer):
    def __init__(self, config, layer_idx=None):
        super().__init__(config)
        self.chunk_size_feed_forward = config.chunk_size_feed_forward
        self.seq_len_dim = 1
        self.attention = BertAttentionScalingNorm(config, is_causal=config.is_decoder, layer_idx=layer_idx)
        self.intermediate = BertIntermediateScalingNorm(config, layer_idx)
        self.output = BertOutputScalingNorm(config)

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: torch.FloatTensor | None = None,
        encoder_hidden_states: torch.FloatTensor | None = None,
        encoder_attention_mask: torch.FloatTensor | None = None,
        past_key_values: Cache | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> torch.Tensor:
        self_attention_output, _ = self.attention(
            hidden_states,
            attention_mask,
            encoder_hidden_states=encoder_hidden_states,
            encoder_attention_mask=encoder_attention_mask,
            past_key_values=past_key_values,
            **kwargs,
        )

        layer_output = apply_chunking_to_forward(
            self.feed_forward_chunk, self.chunk_size_feed_forward, self.seq_len_dim, self_attention_output
        )
        return layer_output

    def feed_forward_chunk(self, attention_output):
        intermediate_output = self.intermediate(attention_output)
        layer_output = self.output(intermediate_output, attention_output)
        return layer_output

class BertEncoderScalingNorm(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.layer = nn.ModuleList([BertLayerScalingNorm(config, layer_idx=i) for i in range(config.num_hidden_layers)])

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: torch.FloatTensor | None = None,
        encoder_hidden_states: torch.FloatTensor | None = None,
        encoder_attention_mask: torch.FloatTensor | None = None,
        past_key_values: Cache | None = None,
        use_cache: bool | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> tuple[torch.Tensor] | BaseModelOutputWithPastAndCrossAttentions:
        for layer_module in self.layer:
            hidden_states = layer_module(
                hidden_states,
                attention_mask,
                encoder_hidden_states=encoder_hidden_states,
                encoder_attention_mask=encoder_attention_mask,
                past_key_values=past_key_values,
                **kwargs,
            )

        return BaseModelOutputWithPastAndCrossAttentions(
            last_hidden_state=hidden_states,
            past_key_values=past_key_values if use_cache else None,
        )

class BertModelScalingNorm(BertModel):
    _no_split_modules = ["BertEmbeddings", "BertLayerScalingNorm"]
    _can_record_outputs = {
        "hidden_states": BertLayerScalingNorm,
        "attentions": BertSelfAttentionScalingNorm,
    }

    def __init__(self, config, add_pooling_layer=True):
        super().__init__(config, add_pooling_layer=add_pooling_layer)
        self.encoder = BertEncoderScalingNorm(config)

class BertForSequenceClassificationScalingNorm(BertForSequenceClassification):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.config = config

        self.bert = BertModelScalingNorm(config)
        classifier_dropout = (
            config.classifier_dropout if config.classifier_dropout is not None else config.hidden_dropout_prob
        )
        self.dropout = nn.Dropout(classifier_dropout)
        self.classifier = nn.Linear(config.hidden_size, config.num_labels)

        self.post_init()

In [14]:
class BertSelfAttentionDeepNorm(BertSelfAttention):
  def __init__(self, config, beta, is_causal=False, layer_idx=None):
        super().__init__(config, is_causal, layer_idx)
        self.value = nn.Linear(config.hidden_size, self.all_head_size)
        self.value.weight.data *= beta

class BertSelfOutputDeepNorm(nn.Module):
    def __init__(self, config, alpha, beta):
        super().__init__()
        self.dense = nn.Linear(config.hidden_size, config.hidden_size)
        self.dense.weight.data *= beta

        self.LayerNorm = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.alpha = alpha

    def forward(self, hidden_states: torch.Tensor, input_tensor: torch.Tensor) -> torch.Tensor:
        hidden_states = self.dense(hidden_states)
        hidden_states = self.dropout(hidden_states)
        hidden_states = self.LayerNorm(hidden_states + self.alpha * input_tensor)
        return hidden_states

class BertAttentionDeepNorm(BertAttention):
    def __init__(self, config, alpha, beta, is_causal=False, layer_idx=None, is_cross_attention=False):
        super().__init__(config, is_causal, layer_idx, is_cross_attention)
        attention_class = BertSelfAttentionDeepNorm
        self.self = attention_class(config, beta, is_causal=is_causal, layer_idx=layer_idx)
        self.output = BertSelfOutputDeepNorm(config, alpha, beta)

class BertIntermediateDeepNorm(BertIntermediate):
    def __init__(self, config, beta):
        super().__init__(config)
        self.dense = nn.Linear(config.hidden_size, config.intermediate_size)
        self.dense.weight.data *= beta

class BertOutputDeepNorm(BertOutput):
    def __init__(self, config, alpha, beta):
        super().__init__(config)
        self.dense = nn.Linear(config.intermediate_size, config.hidden_size)
        self.dense.weight.data *= beta
        self.alpha = alpha

    def forward(self, hidden_states: torch.Tensor, input_tensor: torch.Tensor) -> torch.Tensor:
        hidden_states = self.dense(hidden_states)
        hidden_states = self.dropout(hidden_states)
        hidden_states = self.LayerNorm(hidden_states + self.alpha * input_tensor)
        return hidden_states

class BertLayerDeepNorm(BertLayer):
    def __init__(self, config, alpha, beta, layer_idx=None):
        super().__init__(config)
        self.attention = BertAttentionDeepNorm(config, alpha, beta, is_causal=config.is_decoder, layer_idx=layer_idx)
        self.intermediate = BertIntermediateDeepNorm(config, beta)
        self.output = BertOutputDeepNorm(config, alpha, beta)

class BertEncoderDeepNorm(BertEncoder):
    def __init__(self, config, alpha, beta):
        super().__init__(config)
        self.config = config
        self.layer = nn.ModuleList([BertLayerDeepNorm(config, alpha, beta, layer_idx=i) for i in range(config.num_hidden_layers)])

class BertModelDeepNorm(BertModel):
    _no_split_modules = ["BertEmbeddings", "BertLayerDeepNorm"]
    _can_record_outputs = {
        "hidden_states": BertLayerDeepNorm,
        "attentions": BertSelfAttentionDeepNorm,
    }

    def __init__(self, config, add_pooling_layer=True):
        super().__init__(config, add_pooling_layer)
        alpha = (2 * config.num_hidden_layers) ** 0.25
        beta = 1 / (8 * config.num_hidden_layers) ** 0.25
        self.encoder = BertEncoderDeepNorm(config, alpha, beta)

class BertForSequenceClassificationDeepNorm(BertForSequenceClassification):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.config = config

        self.bert = BertModelDeepNorm(config)
        classifier_dropout = (
            config.classifier_dropout if config.classifier_dropout is not None else config.hidden_dropout_prob
        )
        self.dropout = nn.Dropout(classifier_dropout)
        self.classifier = nn.Linear(config.hidden_size, config.num_labels)

        self.post_init()

In [15]:
import torch
from torch import nn

from transformers.activations import ACT2FN
from transformers.cache_utils import Cache
from transformers.modeling_layers import GradientCheckpointingLayer
from transformers.modeling_outputs import BaseModelOutputWithPastAndCrossAttentions
from transformers.processing_utils import Unpack
from transformers.pytorch_utils import apply_chunking_to_forward
from transformers.utils import TransformersKwargs, auto_docstring
from transformers.models.bert.modeling_bert import BertAttention, BertModel, BertSelfAttention, BertLayer, BertEncoder, BertForSequenceClassification, BertIntermediate

class BertPeriLNFfnOutput(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.dense = nn.Linear(config.intermediate_size, config.hidden_size)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        hidden_states = self.dense(hidden_states)
        hidden_states = self.dropout(hidden_states)
        return hidden_states


class BertPeriLNLayer(BertLayer):
    def __init__(self, config, layer_idx=None):
        super().__init__(config)
        self.chunk_size_feed_forward = config.chunk_size_feed_forward
        self.seq_len_dim = 1
        self.attention = BertAttention(config, is_causal=config.is_decoder, layer_idx=layer_idx)
        self.intermediate = BertIntermediate(config)
        self.ffn_output = BertPeriLNFfnOutput(config)
        self.LayerNorm = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)
        self.LayerNorm = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: torch.FloatTensor | None = None,
        encoder_hidden_states: torch.FloatTensor | None = None,
        encoder_attention_mask: torch.FloatTensor | None = None,
        past_key_values: Cache | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> torch.Tensor:
        self_attention_output, _ = self.attention(
            hidden_states,
            attention_mask,
            past_key_values=past_key_values,
            **kwargs,
        )

        layer_output = apply_chunking_to_forward(
            self.feed_forward_chunk, self.chunk_size_feed_forward, self.seq_len_dim, self_attention_output
        )
        return layer_output

    def feed_forward_chunk(self, attention_output):
        residual = attention_output
        hidden_states = self.pre_feedforward_layernorm(attention_output)
        intermediate_output = self.intermediate(hidden_states)
        ffn_output = self.ffn_output(intermediate_output)
        hidden_states = self.post_feedforward_layernorm(ffn_output) + residual
        return hidden_states


class BertPeriLNEncoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.layer = nn.ModuleList([BertPeriLNLayer(config, layer_idx=i) for i in range(config.num_hidden_layers)])

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: torch.FloatTensor | None = None,
        encoder_hidden_states: torch.FloatTensor | None = None,
        encoder_attention_mask: torch.FloatTensor | None = None,
        past_key_values: Cache | None = None,
        use_cache: bool | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> tuple[torch.Tensor] | BaseModelOutputWithPastAndCrossAttentions:
        for layer_module in self.layer:
            hidden_states = layer_module(
                hidden_states,
                attention_mask,
                past_key_values=past_key_values,
                **kwargs,
            )

        return BaseModelOutputWithPastAndCrossAttentions(
            last_hidden_state=hidden_states,
            past_key_values=past_key_values if use_cache else None,
        )


class BertPeriLNModel(BertModel):
    _no_split_modules = ["BertEmbeddings", "BertPeriLNLayer"]

    def __init__(self, config, add_pooling_layer=True):
        super().__init__(config, add_pooling_layer=add_pooling_layer)
        self.encoder = BertPeriLNEncoder(config)

class BertPeriLNForSequenceClassification(BertForSequenceClassification):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.config = config

        self.bert = BertPeriLNModel(config)
        classifier_dropout = (
            config.classifier_dropout if config.classifier_dropout is not None else config.hidden_dropout_prob
        )
        self.dropout = nn.Dropout(classifier_dropout)
        self.classifier = nn.Linear(config.hidden_size, config.num_labels)

        self.post_init()

In [16]:
from torchmetrics.classification import F1Score

def find_f1(model, tokenizer, dataloader):
    f1 = F1Score(task="binary").to(device)
    model.eval()
    with torch.no_grad():
        for batch in dataloader:
            combined = []
            for ii in range(len(batch['word'])):
                combined.append(f"{batch['word'][ii]} [SEP] {batch['sentence1'][ii]}")
            inputs = tokenizer(combined, batch['sentence2'], return_tensors="pt", padding=True)
            inputs = inputs.to(device)
            labels = batch['label']
            labels = labels.to(device)

            outputs = model(**inputs)
            preds = torch.tensor([0 if o[0] > o[1] else 1 for o in outputs.logits])
            preds = preds.to(device)
            f1(preds, labels)

    common_f1 = f1.compute()
    return common_f1

def train_model_early_stop(model, tokenizer, model_name, train_dataloader, val_dataloader, optimizer, loss_fn, save_model = 0):
    best_val_f1 = 0
    patience = 2
    epochs_no_improve = 0
    for epoch in range(50):
        f1 = F1Score(task="binary").to(device)
        for batch in train_dataloader:
            model.train()
            combined = []
            for ii in range(len(batch['word'])):
                combined.append(f"{batch['word'][ii]} [SEP] {batch['sentence1'][ii]}")
            inputs = tokenizer(combined, batch['sentence2'], return_tensors="pt", padding=True)
            inputs = inputs.to(device)
            labels = batch['label']
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(**inputs)
            logits = outputs.logits
            loss = loss_fn(logits, labels)
            print(loss)

            preds = torch.tensor([0 if o[0] > o[1] else 1 for o in logits])
            preds = preds.to(device)
            f1(preds, labels)
            loss.backward()
            optimizer.step()

        test_f1 = f1.compute()
        val_f1 = find_f1(model, tokenizer, val_dataloader)
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            epochs_no_improve = 0
            if save_model:
                model.save_pretrained(f"/content/drive/MyDrive/colab_results/models/{model_name}")
        else:
            epochs_no_improve += 1

        if epochs_no_improve > patience:
            print("Early stopping")
            break

        print(f"{epoch} epoch: test {str(test_f1)}, validation {str(val_f1)}")
        matr_processing(model, tokenizer, test_sentence1, test_sentence2, f"{model_name}/ang{epoch + 1}")
        grad_processing(model, tokenizer, test_sentence1, test_sentence2, test_label, f"{model_name}/grad{epoch + 1}")
        with open(f'/content/drive/MyDrive/colab_results/{model_name}//F1_{epoch + 1}.txt', 'w', encoding='utf-8') as f:
             f.write(f"test: {str(test_f1)}, validation: {str(val_f1)}")

In [ ]:
set_seed(SEED)
model = BertForSequenceClassificationDeepNorm(config).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-2)

train_model_early_stop(model, tokenizer, "russe/deepbert", train_dataloader, val_dataloader, optimizer, loss_fn)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("DeepPavlov/rubert-base-cased")
set_seed(SEED)
model = BertForSequenceClassificationScalingNorm(config).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-2)

train_model_early_stop(model, tokenizer, "russe/scalingbert(repeat)", train_dataloader, val_dataloader, optimizer, loss_fn)

In [ ]:
SEED = 42

config = BertConfig(
    vocab_size=130000,
    num_hidden_layers=16,
    hidden_size=256,
    num_attention_heads=8,
    max_position_embeddings=1024,
    num_labels=2,
)

tokenizer = AutoTokenizer.from_pretrained("DeepPavlov/rubert-base-cased")

set_seed(SEED)
model = BertForSequenceClassification(config).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-2)

train_model_early_stop(model, tokenizer, "russe/16layers_config/seed42/bert", train_dataloader, val_dataloader, optimizer, loss_fn)

set_seed(SEED)
model = BertForSequenceClassificationPreLayerNorm(config).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-2)

train_model_early_stop(model, tokenizer, "russe/16layers_config/seed42/prebert", train_dataloader, val_dataloader, optimizer, loss_fn)

set_seed(SEED)
model = BertForSequenceClassificationHybridNorm(config).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-2)

train_model_early_stop(model, tokenizer, "russe/16layers_config/seed42/hybridbert", train_dataloader, val_dataloader, optimizer, loss_fn)

set_seed(SEED)
model = BertForSequenceClassificationDeepNorm(config).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-2)

train_model_early_stop(model, tokenizer, "russe/16layers_config/seed42/deepbert", train_dataloader, val_dataloader, optimizer, loss_fn)

set_seed(SEED)
model = BertForSequenceClassificationScalingNorm(config).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-2)

train_model_early_stop(model, tokenizer, "russe/16layers_config/seed42/scalingbert", train_dataloader, val_dataloader, optimizer, loss_fn)

In [ ]:
from google.colab import runtime
runtime.unassign()